In [1]:
import subprocess
subprocess.run(['pip', 'install', 'chromadb', 'sentence-transformers', 'groq', '--quiet'])
print('Dependencies installed')

Dependencies installed


In [2]:
%pip install groq

from dotenv import load_dotenv
import os
import pandas as pd
import numpy as np
import chromadb
from groq import Groq
from sentence_transformers import SentenceTransformer

INSTITUTE       = 'msrit'
CHROMA_PATH     = '../outputs/chroma_db'
COLLECTION_NAME = 'curriculum_intelligence'
EMBED_MODEL     = 'all-MiniLM-L6-v2'
GROQ_MODEL      = 'llama-3.1-8b-instant'
TOP_K           = 8
TOTAL_COMPANIES = 10

load_dotenv()
GROQ_API_KEY = os.environ.get('GROQ_API_KEY')

groq_client = Groq(api_key=GROQ_API_KEY)

try:
    test = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{'role': 'user', 'content': 'reply with the single word: ready'}],
        max_tokens=5
    )
    print(f'Groq connected — model: {GROQ_MODEL}')
    print(f'Test response: {test.choices[0].message.content.strip()}')
except Exception as e:
    print(f'Groq connection failed: {e}')
    print('Check your GROQ_API_KEY')

print('Imports OK')


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


c:\Users\parni\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Groq connected — model: llama-3.1-8b-instant
Test response: ready
Imports OK


In [ ]:
import subprocess
subprocess.run(["pip", "install", "spacy", "--quiet"], check=False)
subprocess.run(["python", "-m", "spacy", "download", "en_core_web_sm", "--quiet"], check=False)

import spacy
nlp_qe = spacy.load("en_core_web_sm", disable=["ner", "parser"])

QUERY_SYNONYMS: dict[str, list[str]] = {
    "ai":          ["machine learning", "deep learning", "nlp", "llm", "generative ai"],
    "ml":          ["machine learning", "deep learning", "mlops"],
    "cloud":       ["aws", "azure", "gcp", "kubernetes", "serverless"],
    "security":    ["vulnerability", "zero trust", "siem", "pentest"],
    "data":        ["sql", "spark", "analytics", "etl", "databricks"],
    "devops":      ["ci/cd", "jenkins", "helm", "ansible", "devsecops"],
    "web":         ["react", "angular", "node.js", "html", "typescript"],
    "mobile":      ["flutter", "swift", "kotlin", "react native"],
    "gap":         ["not taught", "missing", "curriculum gap", "priority"],
    "urgent":      ["highest priority", "debt score", "cli", "overdue"],
    "emerging":    ["upcoming", "watch list", "early warning", "new skill"],
}

INTENT_FILTERS: dict[str, str] = {
    "gap":        "gap_summary",
    "missing":    "gap_summary",
    "curriculum": "syllabus_card",
    "syllabus":   "syllabus_card",
    "watch":      "watch_list",
    "emerging":   "watch_list",
    "upcoming":   "trajectory_summary",
    "trajectory": "trajectory_summary",
}

def expand_query(query: str) -> tuple[str, str | None]:
    """
    Returns:
        expanded_query : original query + synonym expansions joined
        inferred_filter: chroma filter_type inferred from intent, or None
    """
    doc = nlp_qe(query.lower())
    lemmas = [t.lemma_ for t in doc if not t.is_space]

    extra_terms = []
    for lemma in lemmas:
        if lemma in QUERY_SYNONYMS:
            extra_terms.extend(QUERY_SYNONYMS[lemma])
    expanded = query if not extra_terms else query + " " + " ".join(extra_terms)

    inferred_filter = None
    for kw, ftype in INTENT_FILTERS.items():
        if kw in query.lower():
            inferred_filter = ftype
            break

    return expanded, inferred_filter

for test_q in [
    "What AI skills are missing from our curriculum?",
    "Which cloud and DevOps skills have the highest gap?",
    "What should I learn to become a data engineer?",
    "Show me emerging skills on the watch list",
]:
    eq, fi = expand_query(test_q)
    print(f"  Q : {test_q}")
    print(f"  EQ: {eq[:120]}")
    print(f"  Filter inferred: {fi}\n")

print("Query expansion module ready.")


  Q : What AI skills are missing from our curriculum?
  EQ: What AI skills are missing from our curriculum? machine learning deep learning nlp llm generative ai
  Filter inferred: gap_summary

  Q : Which cloud and DevOps skills have the highest gap?
  EQ: Which cloud and DevOps skills have the highest gap? aws azure gcp kubernetes serverless not taught missing curriculum ga
  Filter inferred: gap_summary

  Q : What should I learn to become a data engineer?
  EQ: What should I learn to become a data engineer? sql spark analytics etl databricks
  Filter inferred: None

  Q : Show me emerging skills on the watch list
  EQ: Show me emerging skills on the watch list
  Filter inferred: watch_list

Query expansion module ready.


In [4]:
analysis     = pd.read_csv('../outputs/skill_analysis.csv')
syllabus     = pd.read_csv('../outputs/cleaned_syllabus.csv')
cooccurrence = pd.read_csv('../outputs/skill_cooccurrence.csv')

syllabus['Institute'] = syllabus['Institute'].str.lower().str.strip()

documents = []

for _, row in analysis.iterrows():
    taught     = 'currently taught at MSRIT' if row['Is_Taught_MSRIT'] == 1 \
                 else 'NOT taught at MSRIT — curriculum gap'
    watch      = ' On the Early Warning Watch List.' \
                 if row.get('Watch_List', 0) == 1 else ''
    transition = f" Transitioning: {row['Transition']}." \
                 if row.get('In_Transition', 0) == 1 else ''
    cooc       = f" Co-occurs with: {row['Top_Cooccurring']}." \
                 if pd.notna(row.get('Top_Cooccurring', '')) \
                 and str(row.get('Top_Cooccurring', '')) != '' else ''
    prop       = f" Propagated gap score: {row['Propagated_Gap_Score']:.3f}." \
                 if row.get('Propagated_Gap_Score', 0) > 0 else ''
    inst_gap   = f" Missing from {int(row.get('Institutional_Gap_Score', 0) * 5)}/5 institutes."

    text = (
        f"Skill: {row['Skill']}. "
        f"Trajectory: {row['Trajectory']}. "
        f"Status: {taught}. "
        f"Demanded by {int(row['Company_Spread'])} of {TOTAL_COMPANIES} companies, "
        f"total frequency {int(row['Freq'])}. "
        f"Trend slope: {row['Trend_Slope']:.2f}. "
        f"Velocity: {row['Velocity']:.2f}. "
        f"Curriculum Lag Index: {int(row['CLI'])} years. "
        f"Priority Score: {row['Priority_Score']:.3f}. "
        f"Debt Score: {row['Debt_Score']:.3f}."
        f"{inst_gap}{watch}{transition}{cooc}{prop}"
    )

    documents.append({
        'id': f"skill_{row['Skill'].replace(' ', '_').replace('/', '_')}",
        'text': text,
        'metadata': {
            'type':           'skill_card',
            'skill':          str(row['Skill']),
            'trajectory':     str(row['Trajectory']),
            'is_gap':         int(row['Is_Taught_MSRIT'] == 0),
            'priority_score': float(row['Priority_Score']),
            'debt_score':     float(row['Debt_Score']),
            'cli':            int(row['CLI']),
            'watch_list':     int(row.get('Watch_List', 0)),
        }
    })

for traj in ['Traditional', 'Transitional', 'Upcoming']:
    group     = analysis[analysis['Trajectory'] == traj]
    gap_group = group[group['Is_Taught_MSRIT'] == 0]
    top5      = gap_group.sort_values('Priority_Score', ascending=False)['Skill'].head(5).tolist()
    desc = {
        'Traditional':  'Established skills, may have declining industry relevance.',
        'Transitional': 'Stable skills shifting in how they are applied.',
        'Upcoming':     'Emerging rapidly — highest urgency for curriculum addition.',
    }[traj]
    text = (
        f"Trajectory group: {traj}. {desc} "
        f"{len(group)} total skills, {len(gap_group)} not taught at MSRIT. "
        f"Average CLI: {group['CLI'].mean():.1f} years. "
        f"Average Priority Score: {group['Priority_Score'].mean():.3f}. "
        f"Top gap skills by priority: {', '.join(top5)}."
    )
    documents.append({
        'id': f"trajectory_{traj.lower()}",
        'text': text,
        'metadata': {'type': 'trajectory_summary', 'trajectory': traj}
    })

if 'Watch_List' in analysis.columns and 'Watch_Urgency' in analysis.columns:
    watch_df = analysis[
        (analysis['Watch_List'] == 1) &
        (analysis['Is_Taught_MSRIT'] == 0)
    ].sort_values('Watch_Urgency', ascending=False)

    if len(watch_df) > 0:
        entries = ' | '.join([
            f"{r['Skill']} (urgency {r.get('Watch_Urgency',0):.3f}, "
            f"{int(r['Company_Spread'])} companies, {r['Trajectory']})"
            for _, r in watch_df.iterrows()
        ])
        documents.append({
            'id': 'watch_list_msrit',
            'text': (
                f"Early Warning Watch List for MSRIT — {len(watch_df)} emerging skills "
                f"not yet mainstream but rising fast. Faculty should plan additions now. "
                f"Skills ranked by urgency: {entries}."
            ),
            'metadata': {'type': 'watch_list', 'institute': 'msrit'}
        })

for (institute, course), grp in syllabus.groupby(['Institute', 'Course']):
    skills_list = grp['Skill'].tolist()
    semester    = str(grp['Semester'].iloc[0]) if 'Semester' in grp.columns else 'unknown'
    documents.append({
        'id': f"syllabus_{institute}_{course.replace(' ', '_')}",
        'text': (
            f"Institute: {institute.upper()}. "
            f"Course: {course} (Semester {semester}). "
            f"Skills taught: {', '.join(skills_list)}. "
            f"Total: {len(skills_list)} skills."
        ),
        'metadata': {
            'type':      'syllabus_card',
            'institute': str(institute),
            'course':    str(course),
            'semester':  semester,
        }
    })

gap_df       = analysis[analysis['Is_Taught_MSRIT'] == 0]
top_priority = gap_df.sort_values('Priority_Score', ascending=False).head(10)['Skill'].tolist()
top_debt     = gap_df.sort_values('Debt_Score', ascending=False).head(10)['Skill'].tolist()
in_trans     = gap_df[gap_df['In_Transition'] == 1]['Skill'].tolist() \
               if 'In_Transition' in gap_df.columns else []

documents.append({
    'id': 'gap_summary_msrit',
    'text': (
        f"MSRIT curriculum gap summary. "
        f"{len(gap_df)} of 95 industry skills are not taught at MSRIT. "
        f"Data from {TOTAL_COMPANIES} companies, 2021-2025. "
        f"Top 10 by Priority Score: {', '.join(top_priority)}. "
        f"Top 10 by Debt Score: {', '.join(top_debt)}. "
        f"Skills in transition: {', '.join(in_trans) if in_trans else 'none'}. "
        f"Average CLI: {gap_df['CLI'].mean():.1f} years. "
        f"Total curriculum debt: {gap_df['Debt_Score'].sum():.2f}."
    ),
    'metadata': {'type': 'gap_summary', 'institute': 'msrit'}
})

print(f'Built {len(documents)} document chunks')
print(f'  Skill cards        : {sum(1 for d in documents if d["metadata"]["type"]=="skill_card")}')
print(f'  Trajectory summaries: {sum(1 for d in documents if d["metadata"]["type"]=="trajectory_summary")}')
print(f'  Watch List chunks  : {sum(1 for d in documents if d["metadata"]["type"]=="watch_list")}')
print(f'  Syllabus cards     : {sum(1 for d in documents if d["metadata"]["type"]=="syllabus_card")}')
print(f'  Gap summaries      : {sum(1 for d in documents if d["metadata"]["type"]=="gap_summary")}')

Built 220 document chunks
  Skill cards        : 95
  Trajectory summaries: 3
  Watch List chunks  : 0
  Syllabus cards     : 121
  Gap summaries      : 1


In [ ]:
def enrich_skill_doc(row, base_text: str) -> str:
    """Append NLP-derived fields to a skill card document string."""
    extra_parts = []

    kws = row.get("Skill_Keywords", "")
    if kws and str(kws) not in ("", "nan"):
        extra_parts.append(f"Keywords: {kws}.")

    neighbors = row.get("Semantic_Neighbors", "")
    if neighbors and str(neighbors) not in ("", "nan"):
        extra_parts.append(f"Semantically similar skills: {neighbors}.")

    theme = row.get("Cluster_Theme", "")
    if theme and str(theme) not in ("", "nan", "unknown"):
        extra_parts.append(f"Skill cluster theme: {theme}.")

    nlp_prio = row.get("NLP_Gap_Priority", None)
    if nlp_prio is not None and float(nlp_prio) > 0:
        extra_parts.append(f"NLP-weighted gap priority: {float(nlp_prio):.3f}.")

    if extra_parts:
        return base_text + " " + " ".join(extra_parts)
    return base_text

enriched_count = 0
for doc in documents:
    if doc["metadata"]["type"] != "skill_card":
        continue
    skill_name = doc["metadata"]["skill"]
    match = analysis[analysis["Skill"] == skill_name]
    if len(match) == 0:
        continue
    row = match.iloc[0]
    doc["text"] = enrich_skill_doc(row, doc["text"])
    enriched_count += 1

print(f"Enriched {enriched_count} skill card documents with NLP metadata.")
print("\nSample enriched document:")
sample = next(d for d in documents if d["metadata"]["type"] == "skill_card")
print(sample["text"])


Enriched 95 skill card documents with NLP metadata.

Sample enriched document:
Skill: advanced sql. Trajectory: Upcoming. Status: currently taught at MSRIT. Demanded by 8 of 10 companies, total frequency 8. Trend slope: -4.00. Velocity: -4.00. Curriculum Lag Index: 0 years. Priority Score: 0.304. Debt Score: 0.000. Missing from 2/5 institutes. Co-occurs with: python, data visualization. Keywords: advanced, sql. Semantically similar skills: sql, sql optimization, sql window functions. Skill cluster theme: sql / advanced / optimization / window. NLP-weighted gap priority: 0.358.


In [6]:
print('Loading embedding model (local, no API)...')
embedder = SentenceTransformer(EMBED_MODEL)
print('Model loaded.')

texts = [d['text'] for d in documents]
print(f'Embedding {len(texts)} chunks...')
embeddings = embedder.encode(texts, show_progress_bar=True, batch_size=32)
print(f'Embeddings shape: {embeddings.shape}')

Loading embedding model (local, no API)...


Loading weights: 100%|██████████| 103/103 [00:01<00:00, 73.74it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded.
Embedding 220 chunks...


Batches: 100%|██████████| 7/7 [00:10<00:00,  1.56s/it]

Embeddings shape: (220, 384)


In [7]:
client = chromadb.PersistentClient(path=CHROMA_PATH)

try:
    client.delete_collection(COLLECTION_NAME)
    print('Cleared existing collection.')
except Exception:
    pass

collection = client.create_collection(
    name=COLLECTION_NAME,
    metadata={'hnsw:space': 'cosine'}
)

collection.add(
    ids=[d['id'] for d in documents],
    embeddings=embeddings.tolist(),
    documents=texts,
    metadatas=[d['metadata'] for d in documents]
)

print(f'Indexed {collection.count()} documents into ChromaDB')
print(f'Stored at: {CHROMA_PATH}')

Cleared existing collection.
Indexed 220 documents into ChromaDB
Stored at: ../outputs/chroma_db


In [8]:
def retrieve(query: str, top_k: int = TOP_K, filter_type: str = None) -> list:
    """Embed query locally and retrieve top_k chunks from ChromaDB."""
    query_vec = embedder.encode([query])[0].tolist()
    where     = {'type': filter_type} if filter_type else None

    results = collection.query(
        query_embeddings=[query_vec],
        n_results=top_k,
        where=where,
        include=['documents', 'metadatas', 'distances']
    )

    return [
        {'text': doc, 'metadata': meta, 'distance': dist}
        for doc, meta, dist in zip(
            results['documents'][0],
            results['metadatas'][0],
            results['distances'][0]
        )
    ]


def build_context(query: str, user_type: str = 'staff') -> str:
    """
    Build retrieval context adapted to user type.
    staff   — curriculum level: gaps, debt, priority, syllabus
    student — personal level: what to learn, watch list, skill clusters
    """
    gap_chunks   = retrieve(query, top_k=1, filter_type='gap_summary')
    skill_chunks = retrieve(query, top_k=5, filter_type='skill_card')
    traj_chunks  = retrieve(query, top_k=1, filter_type='trajectory_summary')
    watch_chunks = retrieve(query, top_k=1, filter_type='watch_list')

    if user_type == 'staff':
        syl_chunks = retrieve(query, top_k=3, filter_type='syllabus_card')
        all_chunks = gap_chunks + traj_chunks + skill_chunks + syl_chunks
    else:
        all_chunks = watch_chunks + traj_chunks + skill_chunks

    return '\n\n'.join([f'[{i+1}] {c["text"]}' for i, c in enumerate(all_chunks)])


ctx = build_context('what skills are missing from cloud computing?')
print('Context preview (first 500 chars):')
print(ctx[:500])
print('...')

Context preview (first 500 chars):
[1] MSRIT curriculum gap summary. 52 of 95 industry skills are not taught at MSRIT. Data from 10 companies, 2021-2025. Top 10 by Priority Score: tensorflow, ci/cd, aws, networking, power bi, rest apis, terraform, node.js, next.js, scikit-learn. Top 10 by Debt Score: aws ec2, multithreading, networking, redis caching, spring boot, sql window functions, tableau, vpc configuration, scikit-learn, shell scripting. Skills in transition: next.js, rest apis. Average CLI: 2.2 years. Total curriculum debt
...


In [ ]:
def retrieve_nlp(query: str, top_k: int = TOP_K, filter_type: str = None) -> list:
    """NLP-expanded retrieval with hybrid re-ranking."""
    expanded_q, inferred_filter = expand_query(query)
    active_filter = filter_type or inferred_filter

    query_vec = embedder.encode([expanded_q])[0].tolist()
    where = {"type": active_filter} if active_filter else None

    results = collection.query(
        query_embeddings=[query_vec],
        n_results=top_k,
        where=where,
        include=["documents", "metadatas", "distances"]
    )

    chunks = [
        {"text": doc, "metadata": meta, "distance": dist}
        for doc, meta, dist in zip(
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0]
        )
    ]

    for chunk in chunks:
        sem_score   = 1 - chunk["distance"]
        nlp_prio    = float(chunk["metadata"].get("priority_score", 0))
        chunk["hybrid_score"] = round(0.7 * sem_score + 0.3 * nlp_prio, 4)

    chunks.sort(key=lambda c: c["hybrid_score"], reverse=True)
    return chunks


def build_context_nlp(query: str, user_type: str = "staff") -> str:
    """NLP-enhanced context builder using retrieve_nlp."""
    gap_chunks   = retrieve_nlp(query, top_k=1, filter_type="gap_summary")
    skill_chunks = retrieve_nlp(query, top_k=5, filter_type="skill_card")
    traj_chunks  = retrieve_nlp(query, top_k=1, filter_type="trajectory_summary")
    watch_chunks = retrieve_nlp(query, top_k=1, filter_type="watch_list")

    if user_type == "staff":
        syl_chunks = retrieve_nlp(query, top_k=3, filter_type="syllabus_card")
        all_chunks = gap_chunks + traj_chunks + skill_chunks + syl_chunks
    else:
        all_chunks = watch_chunks + traj_chunks + skill_chunks

    seen, unique = set(), []
    for c in all_chunks:
        key = c["text"][:80]
        if key not in seen:
            seen.add(key)
            unique.append(c)

    return "\n\n".join([
        f"[{i+1}] (hybrid_score={c['hybrid_score']:.3f}) {c['text']}"
        for i, c in enumerate(unique)
    ])


def ask_nlp(query: str, user_type: str = "staff", verbose: bool = False) -> str:
    """Full NLP-augmented RAG pipeline."""
    context = build_context_nlp(query, user_type=user_type)

    if verbose:
        print("=== NLP-EXPANDED CONTEXT ===")
        print(context)
        print("=== END CONTEXT ===\n")

    user_message = (
        f"Use the following retrieved curriculum intelligence data to answer the question.\n"
        f"Answer ONLY from this context. If something is not in the context, "
        f"say: Data not available.\n\n"
        f"--- RETRIEVED CONTEXT ---\n"
        f"{context}\n"
        f"--- END CONTEXT ---\n\n"
        f"Question: {query}"
    )

    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPTS[user_type]},
            {"role": "user",   "content": user_message}
        ],
        temperature=0.2,
        max_tokens=1024
    )
    return response.choices[0].message.content


print("NLP-augmented RAG pipeline ready.")
print(f"  Query expansion : spaCy lemmatisation + {len(QUERY_SYNONYMS)} domain synonym groups")
print(f"  Re-ranking      : 70% semantic similarity + 30% NLP gap priority")
print(f"  De-duplication  : text-hash dedup across retrieval batches")

eq, fi = expand_query("What AI and cloud skills are missing from our curriculum?")
print(f"\nSample expansion: ...{eq[50:130]}...")
print(f"Inferred filter : {fi}")


NLP-augmented RAG pipeline ready.
  Query expansion : spaCy lemmatisation + 11 domain synonym groups
  Re-ranking      : 70% semantic similarity + 30% NLP gap priority
  De-duplication  : text-hash dedup across retrieval batches

Sample expansion: ...iculum? machine learning deep learning nlp llm generative ai aws azure gcp kuber...
Inferred filter : gap_summary


In [10]:
SYSTEM_PROMPTS = {

    'staff': (
        "You are a Curriculum Intelligence Assistant for engineering institutes in Bangalore.\n"
        "You help faculty make data-driven decisions about curriculum updates based on industry skill demand data.\n\n"
        f"You have access to skill demand data from {TOTAL_COMPANIES} companies "
        "(Microsoft, Oracle, SAP Labs, HPE, IBM, Morgan Stanley, NatWest, BT Group, Thales, Zebra Technologies) "
        "over 2021-2025.\n"
        "Key metrics in the data:\n"
        "- CLI (Curriculum Lag Index): years a skill has been demanded before curriculum responds.\n"
        "- Priority Score: urgency to add a skill (0-1, higher = more urgent).\n"
        "- Debt Score: accumulated curriculum lag (higher = more overdue).\n"
        "- Trajectory: Traditional (established), Transitional (shifting), Upcoming (emerging).\n"
        "- Propagated Gap Score: indirectly at-risk skills via co-occurrence graph.\n\n"
        "Rules:\n"
        "- Answer ONLY from the retrieved context provided. Never invent numbers.\n"
        "- Always cite CLI, Priority Score, or Debt Score when recommending skills.\n"
        "- Mention co-occurring skills so faculty can plan clusters, not isolated additions.\n"
        "- Be specific and actionable. Faculty need to justify changes to committees.\n"
        "- If the context does not contain enough information, say: Data not available."
    ),

    'student': (
        "You are a Career Skills Advisor for engineering students in Bangalore.\n"
        "You help students understand what skills to learn based on real industry demand data.\n\n"
        f"You have access to skill demand data from {TOTAL_COMPANIES} major companies over 2021-2025.\n"
        "Key information available:\n"
        "- Which skills are Upcoming (learn now), Transitional (stable), Traditional (established).\n"
        "- A Watch List of skills that will become important soon.\n"
        "- Which skills are commonly demanded together (learn as a cluster).\n\n"
        "Rules:\n"
        "- Give practical, prioritised advice a student can act on today.\n"
        "- Recommend a clear learning order, not just a flat list.\n"
        "- Mention which companies demand each skill so students know the context.\n"
        "- Highlight Watch List skills as early opportunities to stand out.\n"
        "- Keep language simple and encouraging.\n"
        "- Answer ONLY from the retrieved context. If not in context, say: Data not available."
    )
}


def ask(query: str, user_type: str = 'staff', verbose: bool = False) -> str:
    """
    Full RAG pipeline:
    1. Embed query locally (sentence-transformers)
    2. Retrieve relevant chunks from ChromaDB (local)
    3. Build grounded prompt
    4. Generate response via Groq API (LLaMA 3, free)

    user_type: 'staff' or 'student'
    """
    context = build_context(query, user_type=user_type)

    if verbose:
        print('=== RETRIEVED CONTEXT ===')
        print(context)
        print('=== END CONTEXT ===\n')

    user_message = (
        f"Use the following retrieved curriculum intelligence data to answer the question.\n"
        f"Answer ONLY from this context. If something is not in the context, "
        f"say: Data not available.\n\n"
        f"--- RETRIEVED CONTEXT ---\n"
        f"{context}\n"
        f"--- END CONTEXT ---\n\n"
        f"Question: {query}"
    )

    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPTS[user_type]},
            {'role': 'user',   'content': user_message}
        ],
        temperature=0.2,
        max_tokens=1024
    )

    return response.choices[0].message.content


print('RAG pipeline ready.')
print(f'Embeddings : sentence-transformers ({EMBED_MODEL}) — local')
print(f'Vector DB  : ChromaDB — local')
print(f'LLM        : {GROQ_MODEL} via Groq API — free')

RAG pipeline ready.
Embeddings : sentence-transformers (all-MiniLM-L6-v2) — local
Vector DB  : ChromaDB — local
LLM        : llama-3.1-8b-instant via Groq API — free


In [ ]:
for q in [
    "What skills should we add to our 6th semester cloud computing and DevOps elective?",
    "Which skills have the highest curriculum debt at MSRIT and why?",
    "What emerging AI and ML skills are companies demanding that we do not currently teach?",
]:
    print(f"[STAFF] {q}\n")
    print(ask_nlp(q, user_type="staff"))
    print("\n" + "="*80 + "\n")

[STAFF] What skills should we add to our 6th semester cloud computing and DevOps elective?

Based on the retrieved curriculum intelligence data, I recommend adding the following skills to your 6th semester cloud computing and DevOps elective:

1. **Cloud Security (CLI: 0 years, Priority Score: 0.251, Debt Score: 0.000)**: This skill is in high demand by all 10 companies, with a high frequency of 19. It is also co-occurring with other skills like Kubernetes, Multi-Cloud, and Terraform, which are already taught at MSRIT.

2. **Kubernetes (CLI: 0 years, Priority Score: 0.500, Debt Score: 0.000)**: This skill is currently taught at MSRIT, but it is in high demand by all 10 companies, with a high frequency of 45. It is also co-occurring with other skills like Cloud Security, Multi-Cloud, and Terraform.

3. **Multi-Cloud (CLI: 0 years, Priority Score: 0.328, Debt Score: 0.000)**: This skill is in high demand by 7 of 10 companies, with a moderate frequency of 11. It is also co-occurring with 

In [ ]:
for q in [
    "I want to become an ML Engineer. What skills should I focus on and in what order?",
    "What skills should I start learning now before they become mainstream requirements?",
    "If I learn Kubernetes, what other skills should I learn alongside it?",
]:
    print(f"[STUDENT] {q}\n")
    print(ask_nlp(q, user_type="student"))
    print("\n" + "="*80 + "\n")

[STUDENT] I want to become an ML Engineer. What skills should I focus on and in what order?

As an aspiring ML Engineer, I'd recommend focusing on the following skills in this order:

1. **Python**: As a fundamental skill for ML, Python is already taught at MSRIT and is in high demand by 7 of 10 companies. It's a great starting point, and you can build upon it.
2. **Machine Learning (ML)**: Since you want to become an ML Engineer, ML is a must-have skill. It's currently taught at MSRIT and is in high demand by 7 of 10 companies. Focus on developing a strong understanding of ML concepts.
3. **Deep Learning (DL)**: As a subfield of ML, DL is in high demand by 10 of 10 companies. It's a crucial skill for any ML Engineer, and you can build upon your existing ML knowledge.
4. **TensorFlow**: TensorFlow is a popular DL framework, and it's in high demand by 9 of 10 companies. It's a great skill to have, especially since it co-occurs with DL and ML.
5. **Feature Engineering**: Feature engineer

In [ ]:
q = "What are the top priority gaps for MSRIT?"
print(f"[DEBUG - NLP STAFF] {q}\n")
print(ask_nlp(q, user_type="staff", verbose=True))

[DEBUG - NLP STAFF] What are the top priority gaps for MSRIT?

=== NLP-EXPANDED CONTEXT ===
[1] (hybrid_score=0.490) MSRIT curriculum gap summary. 52 of 95 industry skills are not taught at MSRIT. Data from 10 companies, 2021-2025. Top 10 by Priority Score: tensorflow, ci/cd, aws, networking, power bi, rest apis, terraform, node.js, next.js, scikit-learn. Top 10 by Debt Score: aws ec2, multithreading, networking, redis caching, spring boot, sql window functions, tableau, vpc configuration, scikit-learn, shell scripting. Skills in transition: next.js, rest apis. Average CLI: 2.2 years. Total curriculum debt: 323.02.

[2] (hybrid_score=0.463) Trajectory group: Upcoming. Emerging rapidly — highest urgency for curriculum addition. 26 total skills, 15 not taught at MSRIT. Average CLI: 0.6 years. Average Priority Score: 0.293. Top gap skills by priority: next.js, siem tools, model monitoring, shap, multi-cloud strategy.

[3] (hybrid_score=0.647) Skill: spring boot. Trajectory: Transitional. 